### 07_simulate_enroll
Live Camera Enrollment & Identity Check

Flow:

1. **Enrollment**
   - Capture N frames from webcam
   - Detect face, compute embeddings + liveness
   - Build HQ template
   - Append to enrolled gallery as a new user ID (e.g. `n9999999`)
   - Append to FAISS

2. **Identity Check**
   - Capture probe frames from webcam
   - Compute embeddings
   - Compare against gallery (FAISS)
   - Decide Authorized / Unauthorized based on threshold

In [48]:
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import faiss
import sys
import time
import torch
from IPython.display import clear_output, display
import PIL.Image
import importlib

# Add project root
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import scripts
from scripts.liveness import LiveFacePipelineFull


importlib.reload(scripts.build_templates)

from scripts.build_templates import (
    load_gallery_hq,
    enroll_new_user_hq,
    build_user_template_hq,
    ID_COL,
)


In [49]:
DATA_ROOT = "../data_processed/vggface2"
GALLERY_PREFIX = "templates_all_enroll_hq"   # Your main gallery

ID_THRESH = 0.70        # Your chosen threshold
FRAMES_PER_SESSION = 8  # Capture e.g. 8 frames per session
NEW_USER_ID = "n9999999"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

pipe = LiveFacePipelineFull(
    device=DEVICE,
    min_face_side=80,
    min_conf=0.9,
    image_size=160,
)
print("LiveFacePipelineFull device:", DEVICE)



LiveFacePipelineFull device: cuda


In [50]:
templates, templates_map, index, gallery_dir = load_gallery_hq(
    data_root=DATA_ROOT,
    out_name_prefix=GALLERY_PREFIX,
)

print("Loaded gallery from:", gallery_dir)
print("Templates shape:", templates.shape)
print("Enrolled IDs:", templates_map[ID_COL].nunique())


Loaded gallery from: ..\data_processed\vggface2\enrolled_users
Templates shape: (480, 512)
Enrolled IDs: 480


In [51]:
def get_embedding_and_liveness(frame_bgr):
    """
    Use LiveFacePipelineFull to get:
      - embedding (512-d)
      - liveness_score
      - face_box_area
      - face_aligned flag

    Returns:
      emb: (D,) float32 or None if no usable face
      liveness: float
      face_box_area: float
      face_aligned: bool
    """
    info = pipe.process_frame(frame_bgr)  # returns a dict

    emb = info.get("embedding", None)
    if emb is None:
        # no usable face detected
        return None

    return (
        emb,
        float(info.get("liveness_score", 0.0)),
        float(info.get("face_box_area", 0.0)),
        bool(info.get("face_aligned", False)),
    )



In [52]:
# Webcam capture and session embedding function
# For Enrolling User and Verifying Identity
def show_frame_jupyter(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = PIL.Image.fromarray(rgb)
    clear_output(wait=True)
    display(img)

# Capture multiple frames from webcam, return embeddings and liveness scores
def capture_session_embeddings(num_frames=FRAMES_PER_SESSION, session_name="Session", show_preview=False):
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Webcam not available")

    embs = []
    liveness_scores = []
    box_areas = []
    aligned_flags = []

    print(f"[{session_name}] Capturing {num_frames} frames — press 'q' in terminal to stop early (if supported)")

    while len(embs) < num_frames:
        ret, frame = cap.read()
        if not ret:
            print("Failed to read frame")
            break

        # Optional live preview (DISABLED by default for headless envs)
        if show_preview:
            try:
                cv2.imshow(session_name, frame)
                # show_frame_jupyter(frame)
            except Exception:
                # If imshow fails (headless env), disable preview
                show_preview = False

        out = get_embedding_and_liveness(frame)
        if out is not None:
            emb, live, box_area, aligned = out

            # Simple gate: only keep good frames
            if aligned and live >= 0.4:
                embs.append(emb)
                liveness_scores.append(live)
                box_areas.append(box_area)
                aligned_flags.append(aligned)
                print(f"\rCaptured: {len(embs)}/{num_frames}", end="")
            # else: you can log skipped frames if you want

        # Early quit (only works if there is a window, otherwise waitKey just returns -1)
        if show_preview:
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cap.release()
    # Safely try to close any OpenCV windows (no-op in headless envs)
    try:
        cv2.destroyAllWindows()
    except Exception:
        pass
    print()

    if len(embs) == 0:
        print(f"[{session_name}] No usable frames captured.")
        return None, None, None, None

    print(f"[{session_name}] Finished with {len(embs)} frames.")
    return (
        np.stack(embs).astype(np.float32),
        np.array(liveness_scores, dtype=np.float32),
        np.array(box_areas, dtype=np.float32),
        np.array(aligned_flags, dtype=bool),
    )


In [53]:
# Score a single probe embedding against the gallery index
def score_probe_embedding(probe_emb, templates_map, index):
    probe = probe_emb.astype(np.float32)
    probe /= np.linalg.norm(probe) + 1e-12
    probe = probe.reshape(1, -1)

    D, I = index.search(probe, k=1)
    score = float(D[0, 0])
    idx = int(I[0, 0])
    best_id = templates_map.iloc[idx][ID_COL]
    return score, best_id, idx

# Verify identity over a session
def verify_session(threshold=ID_THRESH):
    probe_embs, probe_live, probe_boxes, probe_align = capture_session_embeddings(
        num_frames=FRAMES_PER_SESSION,
        session_name="Verification"
    )
    if probe_embs is None:
        print("No probe embeddings captured.")
        return

    scores = []
    ids = []

    for emb in probe_embs:
        s, pid, idx = score_probe_embedding(emb, templates_map, index)
        scores.append(s)
        ids.append(pid)

    best_i = int(np.argmax(scores))
    best_score = scores[best_i]
    best_id = ids[best_i]

    authorized = best_score >= threshold

    print("\n=== Verification Result ===")
    print("Best score:", best_score)
    print("Matched ID:", best_id)
    print("Matched Index:", idx)
    print("Threshold:", threshold)
    print("Decision:", "AUTHORIZED " if authorized else "UNAUTHORIZED ")

    return {
        "scores": scores,
        "best_score": best_score,
        "best_id": best_id,
        "authorized": authorized,
    }


In [94]:
# ====== Live enrollment=====
print(f"Starting ENROLLMENT for {NEW_USER_ID}")

frame_embs, frame_live, frame_boxes, frame_align = capture_session_embeddings(
    num_frames=FRAMES_PER_SESSION,
    session_name="Enrollment",
    show_preview=True,   # important in this environment
)

if frame_embs is None:
    print("Enrollment failed — no face captured")
else:
    print("Building HQ template and saving...")
    templates, templates_map, index = enroll_new_user_hq(
        new_person_id=NEW_USER_ID,
        frame_embs=frame_embs,
        frame_liveness=frame_live,
        frame_face_box_area=frame_boxes,
        frame_face_aligned=frame_align,
        split="enroll",
        data_root=DATA_ROOT,
        out_name_prefix=GALLERY_PREFIX,
    )

    print("Enrollment complete!")
    print("Gallery size:", templates_map.shape[0])


Starting ENROLLMENT for n9999999
[Enrollment] Capturing 8 frames — press 'q' in terminal to stop early (if supported)
Captured: 8/8
[Enrollment] Finished with 8 frames.
Building HQ template and saving...


2025-12-02 04:40:10,927 INFO [enroll_new_user_hq] Updated existing template for n9999999 at index 480


[enroll_new_user_hq] Enrolled/updated n9999999 at template_index=480
[enroll_new_user_hq] Updated templates: ..\data_processed\vggface2\enrolled_users\templates_all_enroll_hq.npy  shape=(481, 512)
[enroll_new_user_hq] Updated map:       ..\data_processed\vggface2\enrolled_users\templates_all_enroll_hq_map.csv  rows=481)
Enrollment complete!
Gallery size: 481


In [96]:
# Reload gallery to make sure we have latest FAISS + map
templates, templates_map, index, gallery_dir = load_gallery_hq(
    data_root=DATA_ROOT,
    out_name_prefix=GALLERY_PREFIX,
)

print("Starting identity check — show your face to the camera")

# ===== = Identity Verification =====
result = verify_session(ID_THRESH)



Starting identity check — show your face to the camera
[Verification] Capturing 8 frames — press 'q' in terminal to stop early (if supported)
Captured: 8/8
[Verification] Finished with 8 frames.

=== Verification Result ===
Best score: 0.885240912437439
Matched ID: n9999999
Matched Index: 480
Threshold: 0.7
Decision: AUTHORIZED 
